## 2.1 理论计算题

### 题目信息
- 输入图像大小：$ 3 \times 32 \times 32 $（通道数 $C_{\text{in}} = 3$，高 $H_{\text{in}} = 32$，宽 $W_{\text{in}} = 32$）
- 卷积核数量：16（输出通道数 $C_{\text{out}} = 16$）
- 卷积核大小：$ 3 \times 5 \times 5 $（每个核的深度=输入通道数=3，高=5，宽=5）
- 填充（Padding）：$P = 2$
- 步幅（Stride）：$S = 2$

---

### 1. 特征图尺寸计算

输出特征图的高度 $H_{\text{out}}$ 和宽度 $W_{\text{out}}$ 公式为：

$$
H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2P - K}{S} \right\rfloor + 1
$$

$$
W_{\text{out}} = \left\lfloor \frac{W_{\text{in}} + 2P - K}{S} \right\rfloor + 1
$$

其中 $K = 5$。

代入数值：

$$
H_{\text{out}} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
$$

$$
W_{\text{out}} = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = 15 + 1 = 16
$$

输出特征图尺寸为：

$$
\boxed{16 \times 16 \times 16}
$$

即：通道数 $16$，高 $16$，宽 $16$。

---

### 2. 单个输出通道的一个像素值的点乘次数

对于一个输出通道的一个像素点，其计算来自：
- 卷积核大小：$3 \times 5 \times 5 = 75$ 个权重
- 每个权重与输入图像对应位置的像素值做一次乘法（忽略加法）

因此，**点乘（乘法）次数 = 卷积核的参数量**。

$$
\boxed{75}
$$

In [2]:
import numpy as np

def max_pool2d(x, kernel_size, stride=None, padding=0):
    # 处理参数
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    
    if stride is None:
        sh = sw = kh
    elif isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    batch, ch, h, w = x.shape
    
    # 填充
    x_pad = np.pad(x, ((0, 0), (0, 0), (ph, ph), (pw, pw)), mode='constant')
    
    # 计算输出尺寸
    out_h = (h + 2 * ph - kh) // sh + 1
    out_w = (w + 2 * pw - kw) // sw + 1
    
    # 初始化输出
    out = np.zeros((batch, ch, out_h, out_w))
    
    # 池化操作
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            window = x_pad[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))
    
    return out


# 示例测试
if __name__ == "__main__":
    # 创建一个随机输入：batch=2, channels=3, height=32, width=32
    x = np.random.randn(2, 3, 32, 32)
    
    # 池化：窗口大小2x2，步幅2，填充0
    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {out.shape}")

输入形状: (2, 3, 32, 32)
输出形状: (2, 3, 16, 16)


## 3.1 理论计算题

### 题目信息
- 输入和输出特征图的通道数均为 $C$
- 卷积层不带偏置

---

### 1. 一个 $5 \times 5$ 卷积层的参数量

对于一个 $5 \times 5$ 卷积层：
- 每个卷积核大小：$5 \times 5 = 25$ 个权重
- 输入通道数为 $C$，因此每个输出通道对应的卷积核深度为 $C$，参数量为 $C \times 25$
- 输出通道数为 $C$，因此总参数量为：

$$
\boxed{25C^2}
$$

---

### 2. 两个串联的 $3 \times 3$ 卷积层的总参数量

对于第一个 $3 \times 3$ 卷积层：
- 每个卷积核大小：$3 \times 3 = 9$ 个权重
- 输入通道数为 $C$，输出通道数也为 $C$，参数量为 $C \times 9 \times C = 9C^2$

对于第二个 $3 \times 3$ 卷积层：
- 输入通道数为 $C$，输出通道数为 $C$，参数量同样为 $9C^2$

两个卷积层串联的总参数量：

$$
\boxed{18C^2}
$$

---

### 结论

使用两个 $3 \times 3$ 卷积级联（参数量 $18C^2$）替代一个 $5 \times 5$ 卷积（参数量 $25C^2$），参数量减少了：

$$
25C^2 - 18C^2 = 7C^2
$$

相对减少比例为 $\frac{7}{25} = 28\%$，同时两个 $3 \times 3$ 卷积能提供更多的非线性变换。

In [3]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            # 第一个 1x1 卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个 1x1 卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)


# 示例测试
if __name__ == "__main__":
    # 创建一个 NiN 块：输入通道3，输出通道16，卷积核3x3，步幅1，填充1
    nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    
    # 测试输入：batch=4, channels=3, height=32, width=32
    x = torch.randn(4, 3, 32, 32)
    out = nin_block(x)
    
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {out.shape}")

输入形状: torch.Size([4, 3, 32, 32])
输出形状: torch.Size([4, 16, 32, 32])


## 4.1 理论计算题

### 题目信息
- 4个样本的特征值：$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$
- 缩放参数：$\gamma = 2$
- 平移参数：$\beta = 1$
- 常数：$\epsilon = 0$

---

### 计算步骤

#### 1. 计算均值 $\mu$

$$
\mu = \frac{1}{4} \sum_{i=1}^{4} x_i = \frac{2 + 4 + 6 + 8}{4} = \frac{20}{4} = 5
$$

#### 2. 计算方差 $\sigma^2$

$$
\sigma^2 = \frac{1}{4} \sum_{i=1}^{4} (x_i - \mu)^2
$$

$$
= \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4}
$$

$$
= \frac{(-3)^2 + (-1)^2 + (1)^2 + (3)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = \frac{20}{4} = 5
$$

#### 3. 标准化（$\epsilon = 0$）

$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}
$$

计算各样本：
- $\hat{x}_1 = \frac{2 - 5}{\sqrt{5}} = \frac{-3}{\sqrt{5}} = -\frac{3}{\sqrt{5}}$
- $\hat{x}_2 = \frac{4 - 5}{\sqrt{5}} = \frac{-1}{\sqrt{5}} = -\frac{1}{\sqrt{5}}$
- $\hat{x}_3 = \frac{6 - 5}{\sqrt{5}} = \frac{1}{\sqrt{5}}$
- $\hat{x}_4 = \frac{8 - 5}{\sqrt{5}} = \frac{3}{\sqrt{5}}$

#### 4. 缩放和平移

$$
y_i = \gamma \hat{x}_i + \beta = 2 \hat{x}_i + 1
$$

代入计算：
- $y_1 = 2 \times \left(-\frac{3}{\sqrt{5}}\right) + 1 = -\frac{6}{\sqrt{5}} + 1$
- $y_2 = 2 \times \left(-\frac{1}{\sqrt{5}}\right) + 1 = -\frac{2}{\sqrt{5}} + 1$
- $y_3 = 2 \times \left(\frac{1}{\sqrt{5}}\right) + 1 = \frac{2}{\sqrt{5}} + 1$
- $y_4 = 2 \times \left(\frac{3}{\sqrt{5}}\right) + 1 = \frac{6}{\sqrt{5}} + 1$

---

### 最终结果

$$
\boxed{y_1 = -\frac{6}{\sqrt{5}} + 1,\quad y_2 = -\frac{2}{\sqrt{5}} + 1,\quad y_3 = \frac{2}{\sqrt{5}} + 1,\quad y_4 = \frac{6}{\sqrt{5}} + 1}
$$

若取近似值（$\sqrt{5} \approx 2.236$）：

$$
y_1 \approx -1.683,\quad y_2 \approx 0.106,\quad y_3 \approx 1.894,\quad y_4 \approx 3.683
$$

In [4]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个卷积层：3x3，步幅为 stride
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层：3x3，步幅为1
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1x1 卷积层（用于调整输入形状，使输入能与输出相加）
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
    
    def forward(self, x):
        # 保存输入用于残差连接
        identity = x
        
        # 第一个卷积块
        out = self.conv1(x)
        out = self.bn1(out)
        out = nn.ReLU()(out)
        
        # 第二个卷积块
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 如果需要，调整输入的形状
        if self.conv3 is not None:
            identity = self.conv3(x)
        
        # 残差连接（按元素相加）
        out += identity
        out = nn.ReLU()(out)
        
        return out


# 示例测试
if __name__ == "__main__":
    # 测试1：输入输出通道相同，不使用 1x1 卷积
    block1 = Residual(in_channels=16, out_channels=16, use_1x1conv=False)
    x1 = torch.randn(4, 16, 32, 32)
    out1 = block1(x1)
    print(f"测试1 - 输入形状: {x1.shape}, 输出形状: {out1.shape}")
    
    # 测试2：输入输出通道不同，使用 1x1 卷积
    block2 = Residual(in_channels=3, out_channels=16, use_1x1conv=True, stride=2)
    x2 = torch.randn(4, 3, 32, 32)
    out2 = block2(x2)
    print(f"测试2 - 输入形状: {x2.shape}, 输出形状: {out2.shape}")

测试1 - 输入形状: torch.Size([4, 16, 32, 32]), 输出形状: torch.Size([4, 16, 32, 32])
测试2 - 输入形状: torch.Size([4, 3, 32, 32]), 输出形状: torch.Size([4, 16, 16, 16])


## 5.1 理论计算题

### 1. 为什么底层特征提取层用小学习率或冻结，顶层输出层用大学习率？

**原因如下：**

- **底层特征提取层**：在源数据集（如 ImageNet）上预训练后，这些层已经学习到了通用的低级特征（如边缘、纹理、颜色等）。这些特征在不同视觉任务中具有较好的泛化能力，不需要大幅调整。因此：
  - 使用**较小的学习率**可以微调这些特征，避免破坏已学到的有用表示
  - 或者直接**冻结**参数，保持特征提取器不变，减少过拟合风险

- **顶层输出层**：该层通常是新初始化的分类层，需要适应目标数据集的类别分布。由于没有预训练知识，需要**较大的学习率**快速学习新类别的决策边界。

这种策略既能保留预训练模型的通用特征提取能力，又能高效适配新任务。

---

### 2. 目标数据集非常小且与源数据集非常相似时的微调策略

**建议策略：**

1. **冻结大部分底层特征提取层**：将预训练模型的卷积基（除最后的全连接层外）全部冻结，仅训练新添加的分类层

2. **使用非常小的学习率**：如果必须微调部分底层，应使用极小的学习率（如 $10^{-5}$ 或更小）

3. **只微调最后几层**：可以选择解冻最后 1-2 个卷积块，其他层保持冻结

4. **使用更强的正则化**：
   - 增加 Dropout 比例
   - 使用权重衰减（Weight Decay）
   - 使用数据增广（即使数据量小）

5. **早停（Early Stopping）**：监控验证集性能，防止过拟合

6. **避免从头训练**：必须使用预训练模型，不能从头开始训练

In [5]:
import torchvision.transforms as transforms
from PIL import Image
import torch

# 定义图像增广管道
transform_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放到 224x224
    #    scale=(0.08, 1.0) 表示裁剪面积占原图面积的 8% 到 100%
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0)),
    
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度、饱和度，变化范围 0.5
    #    brightness=0.5, contrast=0.5, saturation=0.5 表示调整因子在 [0.5, 1.5] 之间随机
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor()
])


# 示例测试
if __name__ == "__main__":
    # 创建一个示例图像（假设为 RGB 图像）
    # 实际使用时替换为 Image.open("your_image.jpg")
    sample_image = Image.new('RGB', (480, 640), color='red')
    
    # 应用增广管道
    augmented_tensor = transform_pipeline(sample_image)
    
    print(f"原始图像尺寸: {sample_image.size}")
    print(f"增广后张量形状: {augmented_tensor.shape}")  # 应为 (3, 224, 224)
    print(f"张量值范围: [{augmented_tensor.min():.4f}, {augmented_tensor.max():.4f}]")
    
    # 多次应用展示随机性
    print("\n多次应用同一图像的结果形状:")
    for i in range(3):
        out = transform_pipeline(sample_image)
        print(f"  第{i+1}次: {out.shape}")

原始图像尺寸: (480, 640)
增广后张量形状: torch.Size([3, 224, 224])
张量值范围: [0.0627, 0.3647]

多次应用同一图像的结果形状:
  第1次: torch.Size([3, 224, 224])
  第2次: torch.Size([3, 224, 224])
  第3次: torch.Size([3, 224, 224])


## 6.1 理论计算题

### 题目信息
- 真实框（Ground Truth）$A = [10, 10, 50, 50]$，格式为 $[x_{\text{min}}, y_{\text{min}}, x_{\text{max}}, y_{\text{max}}]$
- 预测框（Prediction Box）$B = [30, 30, 70, 70]$

---

### 1. 计算交集区域

交集区域的左上角坐标：
$$
x_{\text{min}}^{\text{inter}} = \max(x_{\text{min}}^A, x_{\text{min}}^B) = \max(10, 30) = 30
$$
$$
y_{\text{min}}^{\text{inter}} = \max(y_{\text{min}}^A, y_{\text{min}}^B) = \max(10, 30) = 30
$$

交集区域的右下角坐标：
$$
x_{\text{max}}^{\text{inter}} = \min(x_{\text{max}}^A, x_{\text{max}}^B) = \min(50, 70) = 50
$$
$$
y_{\text{max}}^{\text{inter}} = \min(y_{\text{max}}^A, y_{\text{max}}^B) = \min(50, 70) = 50
$$

交集区域的宽度和高度：
$$
w_{\text{inter}} = x_{\text{max}}^{\text{inter}} - x_{\text{min}}^{\text{inter}} = 50 - 30 = 20
$$
$$
h_{\text{inter}} = y_{\text{max}}^{\text{inter}} - y_{\text{min}}^{\text{inter}} = 50 - 30 = 20
$$

交集面积：
$$
S_{\text{inter}} = w_{\text{inter}} \times h_{\text{inter}} = 20 \times 20 = 400
$$

---

### 2. 计算并集区域

真实框 $A$ 的面积：
$$
S_A = (50 - 10) \times (50 - 10) = 40 \times 40 = 1600
$$

预测框 $B$ 的面积：
$$
S_B = (70 - 30) \times (70 - 30) = 40 \times 40 = 1600
$$

并集面积：
$$
S_{\text{union}} = S_A + S_B - S_{\text{inter}} = 1600 + 1600 - 400 = 2800
$$

---

### 3. 计算 IoU

$$
\text{IoU} = \frac{S_{\text{inter}}}{S_{\text{union}}} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857
$$

---

### 最终结果

$$
\boxed{\text{IoU} = \frac{1}{7} \approx 0.1429}
$$

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    
    def forward(self, pred, target):
        num_classes = pred.shape[-1]
        
        # 创建平滑后的目标分布
        # 真实类别概率为 1 - epsilon，其余类别概率为 epsilon / (K - 1)
        smooth_target = torch.zeros_like(pred)
        smooth_target.fill_(self.epsilon / (num_classes - 1))
        smooth_target.scatter_(1, target.unsqueeze(1), 1 - self.epsilon)
        
        # 计算交叉熵损失
        log_probs = F.log_softmax(pred, dim=-1)
        loss = -torch.sum(smooth_target * log_probs, dim=-1)
        
        # 规约
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


# 更简洁的函数式版本
def label_smoothing_cross_entropy(pred, target, epsilon=0.1, reduction='mean'):
    num_classes = pred.shape[-1]
    
    # 计算 log_softmax
    log_probs = F.log_softmax(pred, dim=-1)
    
    # 创建平滑标签
    smooth_target = torch.full_like(pred, epsilon / (num_classes - 1))
    smooth_target.scatter_(1, target.unsqueeze(1), 1 - epsilon)
    
    # 计算损失
    loss = -torch.sum(smooth_target * log_probs, dim=-1)
    
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss


# 示例测试
if __name__ == "__main__":
    # 设置随机种子
    torch.manual_seed(42)
    
    # 模拟数据：batch_size=4，类别数=10
    batch_size, num_classes = 4, 10
    pred = torch.randn(batch_size, num_classes)
    target = torch.tensor([2, 5, 1, 7])  # 真实标签
    
    # 使用类版本
    criterion = LabelSmoothingCrossEntropy(epsilon=0.1, reduction='mean')
    loss1 = criterion(pred, target)
    
    # 使用函数式版本
    loss2 = label_smoothing_cross_entropy(pred, target, epsilon=0.1, reduction='mean')
    
    # 对比标准交叉熵
    standard_ce = nn.CrossEntropyLoss()(pred, target)
    
    print(f"预测值形状: {pred.shape}")
    print(f"真实标签: {target}")
    print(f"\n标签平滑交叉熵损失 (类版本): {loss1.item():.6f}")
    print(f"标签平滑交叉熵损失 (函数式版本): {loss2.item():.6f}")
    print(f"标准交叉熵损失: {standard_ce.item():.6f}")
    
    # 验证平滑标签
    smooth_labels = torch.full((num_classes,), 0.1 / 9)  # epsilon/(K-1) = 0.1/9 ≈ 0.0111
    smooth_labels[2] = 0.9
    print(f"\n示例平滑标签 (类别0-9):")
    print(smooth_labels)

预测值形状: torch.Size([4, 10])
真实标签: tensor([2, 5, 1, 7])

标签平滑交叉熵损失 (类版本): 2.430207
标签平滑交叉熵损失 (函数式版本): 2.430207
标准交叉熵损失: 2.375969

示例平滑标签 (类别0-9):
tensor([0.0111, 0.0111, 0.9000, 0.0111, 0.0111, 0.0111, 0.0111, 0.0111, 0.0111,
        0.0111])
